# 03_02 — Thai Sign Language Model Training (LSTM / GRU, extended naming)
Supports: `processed` (keypoint) · `oldprocessed` (demo keypoint) · `cutting` (video) · LSTM / GRU

Output directory format: `{lr}_{batch}_{inputtype}_{modeltype}_{dropout}_{l2}_{labelsmoothing}_{units}`

## 0 · Imports & environment

In [1]:
import os, json, time, random, warnings, datetime
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_fscore_support, top_k_accuracy_score
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, mixed_precision

warnings.filterwarnings('ignore')
print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))

for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

TensorFlow: 2.16.1
GPU devices: []


## 1 · Configuration — edit here to switch experiments

In [2]:
# INPUT_TYPE : 'processed'    → numerical keypoint .npy files (experiment_model/datasets/processed)
#              'oldprocessed' → keypoint .npy files from demo_data/processed
#              'cutting'      → video frames from datasets/cutting/ directory
INPUT_TYPE   = 'oldprocessed'

# MODEL_TYPE : 'lstm' | 'gru'
MODEL_TYPE   = 'lstm'

BATCH_SIZE   = 8             # choices: 8 | 16 | 32
LEARNING_RATE = 1e-3          # choices: 1e-3 | 1e-4 | 1e-5 | 1e-6 | 1e-7

LSTM_UNITS     = [128, 64, 32]
DROPOUT_RATE   = 0.2
L2_REG         = 1e-3
LABEL_SMOOTHING = 0.05
MIXUP_ALPHA    = 0.1
EPOCHS         = 200

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

ROOT          = Path('..').resolve()
DATA_DIR      = ROOT / 'datasets'
DEMO_DATA_DIR = ROOT.parent / 'demo_data'
MODELS_DIR    = ROOT / 'models_02'
LOGS_DIR      = ROOT / 'logs'

lr_exp     = int(round(np.log10(LEARNING_RATE)))
units_str  = '-'.join(str(u) for u in LSTM_UNITS)
EXP_NAME   = f'{lr_exp}_{BATCH_SIZE}_{INPUT_TYPE}_{MODEL_TYPE}_{DROPOUT_RATE}_{L2_REG}_{LABEL_SMOOTHING}_{units_str}'
EXP_DIR    = MODELS_DIR / EXP_NAME
LOG_FILE   = LOGS_DIR / 'training' / f'{EXP_NAME}.json'

EXP_DIR.mkdir(parents=True, exist_ok=True)
(LOGS_DIR / 'training').mkdir(parents=True, exist_ok=True)

print(f'Experiment : {EXP_NAME}')
print(f'Output dir : {EXP_DIR}')

Experiment : -3_8_oldprocessed_lstm_0.2_0.001_0.05_128-64-32
Output dir : C:\Users\Napat\termtemsl\experiment_model\models_02\-3_8_oldprocessed_lstm_0.2_0.001_0.05_128-64-32


## 2 · Data preparation

In [3]:
if INPUT_TYPE == 'processed':
    label_path = DATA_DIR / 'processed' / 'class_labels.json'
    with open(label_path) as f:
        label_info = json.load(f)
    CLASSES      = label_info['classes']
    label_to_idx = label_info['class_to_label']
    idx_to_label = {int(k): v for k, v in label_info['label_to_class'].items()}

    with open(DATA_DIR / 'processed' / 'feature_config.json') as f:
        feat_cfg = json.load(f)
else:
    # 'oldprocessed' and 'cutting' both use labels from demo_data/processed
    label_path = DEMO_DATA_DIR / 'processed' / 'labels.json'
    with open(label_path) as f:
        label_info = json.load(f)
    CLASSES      = label_info['classes']
    label_to_idx = label_info['class_to_idx']
    idx_to_label = {v: k for k, v in label_info['class_to_idx'].items()}

NUM_CLASSES = len(CLASSES)
print(f'Classes ({NUM_CLASSES}):', CLASSES)

Classes (10): ['ขอโทษ', 'ดี', 'สวัสดี', 'อะไร', 'เงิน', 'โทรศัพท์', 'ใคร', 'ใช่', 'ไม่', 'ไม่ดี']


In [ ]:
def load_processed_data():
    """Load pre-extracted keypoint numpy arrays from experiment_model/datasets/processed."""
    proc = DATA_DIR / 'processed'
    X_train = np.load(proc / 'X_train.npy').astype(np.float32)
    X_val   = np.load(proc / 'X_val.npy').astype(np.float32)
    X_test  = np.load(proc / 'X_test.npy').astype(np.float32)
    y_train = np.load(proc / 'y_train.npy')
    y_val   = np.load(proc / 'y_val.npy')
    y_test  = np.load(proc / 'y_test.npy')
    print(f'X_train {X_train.shape}  X_val {X_val.shape}  X_test {X_test.shape}')
    return X_train, X_val, X_test, y_train, y_val, y_test


def load_oldprocessed_data():
    """Load pre-extracted keypoint numpy arrays from demo_data/processed."""
    proc = DEMO_DATA_DIR / 'processed'
    X_train = np.load(proc / 'X_train.npy').astype(np.float32)
    X_val   = np.load(proc / 'X_val.npy').astype(np.float32)
    X_test  = np.load(proc / 'X_test.npy').astype(np.float32)
    y_train = np.load(proc / 'y_train.npy')
    y_val   = np.load(proc / 'y_val.npy')
    y_test  = np.load(proc / 'y_test.npy')
    print(f'X_train {X_train.shape}  X_val {X_val.shape}  X_test {X_test.shape}')
    return X_train, X_val, X_test, y_train, y_val, y_test


def load_video_data(target_frames=30, img_size=64):
    """
    Load videos from datasets/cutting/ using the train/val/test split defined by
    demo_data/processed/keypoints/{split}/*.npy.
    Augmented keypoint files (_a1/_a2/_a3) are skipped — no corresponding video.
    '_base' keypoint files map to '{class}.mp4' (the unnumbered recording).
    """
    import cv2, re

    cutting_dir = DATA_DIR / 'cutting'
    kp_dir      = DEMO_DATA_DIR / 'processed' / 'keypoints'

    def read_video(path):
        cap = cv2.VideoCapture(str(path))
        frames = []
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.resize(frame, (img_size, img_size))
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
        cap.release()
        if len(frames) == 0:
            return np.zeros((target_frames, img_size, img_size, 3), dtype=np.float32)
        frames = np.array(frames, dtype=np.float32) / 255.0
        idx = np.linspace(0, len(frames) - 1, target_frames).astype(int)
        return frames[idx]

    def kp_stem_to_video(stem):
        """Return video filename for a keypoint stem, or None if augmented."""
        if re.search(r'_a\d+$', stem):
            return None
        if stem.endswith('_base'):
            return stem[:-5] + '.mp4'
        return stem + '.mp4'

    def class_from_stem(stem):
        """Extract class name from keypoint stem (strips _base / _NN / _aN suffixes)."""
        stem = re.sub(r'_a\d+$', '', stem)
        stem = re.sub(r'_base$', '', stem)
        stem = re.sub(r'_\d+$', '', stem)
        return stem

    splits = {}
    for split in ('train', 'val', 'test'):
        pairs = []
        for npy in sorted((kp_dir / split).glob('*.npy')):
            vid_name = kp_stem_to_video(npy.stem)
            if vid_name is None:
                continue
            cls = class_from_stem(npy.stem)
            if cls not in label_to_idx:
                continue
            vid_path = cutting_dir / vid_name
            if not vid_path.exists():
                continue
            seq = read_video(vid_path)
            pairs.append((seq, label_to_idx[cls]))
        splits[split] = pairs

    def to_arrays(pairs):
        X = np.stack([p[0] for p in pairs])
        y = np.array([p[1] for p in pairs])
        return X, y

    X_train, y_train = to_arrays(splits['train'])
    X_val,   y_val   = to_arrays(splits['val'])
    X_test,  y_test  = to_arrays(splits['test'])
    print(f'Video X_train {X_train.shape}  X_val {X_val.shape}  X_test {X_test.shape}')
    return X_train, X_val, X_test, y_train, y_val, y_test


if INPUT_TYPE == 'processed':
    X_train, X_val, X_test, y_train, y_val, y_test = load_processed_data()
elif INPUT_TYPE == 'oldprocessed':
    X_train, X_val, X_test, y_train, y_val, y_test = load_oldprocessed_data()
else:  # cutting
    X_train, X_val, X_test, y_train, y_val, y_test = load_video_data()

INPUT_SHAPE = X_train.shape[1:]
print('Input shape:', INPUT_SHAPE)
print('Label distribution (train):', np.bincount(y_train.astype(int)))

X_train (1200, 130, 404)  X_val (290, 130, 404)  X_test (10, 130, 404)
Input shape: (130, 404)
Label distribution (train): [120 120 120 120 120 120 120 120 120 120]


In [ ]:
def mixup_batch(X, y, alpha=MIXUP_ALPHA, num_classes=NUM_CLASSES):
    """Apply mixup to a batch; y must be one-hot. Works for any input shape."""
    if alpha <= 0:
        return X, y
    lam_shape = (len(X),) + (1,) * (X.ndim - 1)
    lam  = np.random.beta(alpha, alpha, size=lam_shape)
    idx  = np.random.permutation(len(X))
    X_mix = lam * X + (1 - lam) * X[idx]
    lam2  = lam.reshape(len(X), 1)
    y_mix = lam2 * y + (1 - lam2) * y[idx]
    return X_mix.astype(np.float32), y_mix.astype(np.float32)


y_train_oh = tf.keras.utils.to_categorical(y_train, NUM_CLASSES)
y_val_oh   = tf.keras.utils.to_categorical(y_val,   NUM_CLASSES)
y_test_oh  = tf.keras.utils.to_categorical(y_test,  NUM_CLASSES)

X_train_m, y_train_m = mixup_batch(X_train, y_train_oh)

train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train_m, y_train_m))
    .shuffle(len(X_train_m), seed=SEED)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
val_ds = (
    tf.data.Dataset.from_tensor_slices((X_val, y_val_oh))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
print('tf.data pipelines ready')

tf.data pipelines ready


## 3 · Model builder

In [6]:
def build_recurrent_model(
    input_shape,
    num_classes,
    model_type='lstm',
    units=None,
    dropout_rate=DROPOUT_RATE,
    l2=L2_REG,
):
    """
    Build LSTM / GRU sequence classifier.
    For video input (5-D) a TimeDistributed CNN stem is prepended.
    """
    if units is None:
        units = LSTM_UNITS

    cell_map = {
        'lstm': layers.LSTM,
        'gru':  layers.GRU,
    }
    assert model_type in cell_map, f'Unknown model_type {model_type!r} — must be lstm or gru'
    Cell = cell_map[model_type]
    reg  = regularizers.l2(l2)

    inp = keras.Input(shape=input_shape, name='input')
    x   = inp

    if len(input_shape) == 4:
        x = layers.TimeDistributed(
            keras.Sequential([
                layers.Conv2D(32, 3, activation='relu', padding='same'),
                layers.MaxPooling2D(2),
                layers.Conv2D(64, 3, activation='relu', padding='same'),
                layers.GlobalAveragePooling2D(),
            ]), name='cnn_stem'
        )(x)

    x = layers.BatchNormalization()(x)

    for i, u in enumerate(units):
        is_last = (i == len(units) - 1)
        x = Cell(
            u,
            return_sequences=not is_last,
            dropout=dropout_rate * 0.5,
            recurrent_dropout=dropout_rate * 0.3,
            kernel_regularizer=reg,
            name=f'{model_type}_{i}',
        )(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(dropout_rate)(x)

    x   = layers.Dense(64, activation='relu', kernel_regularizer=reg)(x)
    x   = layers.Dropout(dropout_rate * 0.5)(x)
    out = layers.Dense(num_classes, activation='softmax', name='output')(x)

    model = keras.Model(inp, out, name=f'{model_type}_classifier')
    return model


model = build_recurrent_model(
    input_shape=INPUT_SHAPE,
    num_classes=NUM_CLASSES,
    model_type=MODEL_TYPE,
)
model.summary()

Model: "lstm_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 130, 404)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 130, 404)       │         1,616 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_0 (LSTM)                   │ (None, 130, 128)       │       272,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 130, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 130, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 130, 64)        │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 130, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 130, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 339,994 (1.30 MB)

 Trainable params: 338,738 (1.29 MB)

 Non-trainable params: 1,256 (4.91 KB)

## 4 · Callbacks & compile

In [7]:
class LearningRateLogger(keras.callbacks.Callback):
    """Appends current LR to history as 'lr'."""
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        logs['lr'] = float(keras.backend.get_value(self.model.optimizer.learning_rate))


def make_callbacks(exp_dir: Path, monitor='val_accuracy'):
    """Factory — add extra callbacks here for future sweeps."""
    ckpt_path = str(exp_dir / 'best_weights.weights.h5')
    return [
        keras.callbacks.EarlyStopping(
            monitor=monitor,
            patience=30,
            restore_best_weights=True,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor=monitor,
            factor=0.5,
            patience=10,
            min_lr=1e-8,
            verbose=1,
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=ckpt_path,
            monitor=monitor,
            save_best_only=True,
            save_weights_only=True,
            verbose=0,
        ),
        LearningRateLogger(),
    ]


loss_fn = keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=loss_fn,
    metrics=[
        'accuracy',
        keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc'),
    ],
)

callbacks = make_callbacks(EXP_DIR)
print('Callbacks ready:', [type(c).__name__ for c in callbacks])

Callbacks ready: ['EarlyStopping', 'ReduceLROnPlateau', 'ModelCheckpoint', 'LearningRateLogger']


## 5 · Train

In [8]:
t0 = time.time()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

TRAIN_TIME = time.time() - t0
print(f'\nTraining time: {TRAIN_TIME/60:.1f} min')

Epoch 1/200
150/150 ━━━━━━━━━━━━━━━━━━━━ 44s 206ms/step - accuracy: 0.1533 - loss: 3.0851 - top3_acc: 0.3958 - val_accuracy: 0.1034 - val_loss: 3.0359 - val_top3_acc: 0.2586 - learning_rate: 0.0010 - lr: 0.0010
Epoch 2/200
150/150 ━━━━━━━━━━━━━━━━━━━━ 32s 212ms/step - accuracy: 0.2508 - loss: 2.8179 - top3_acc: 0.5250 - val_accuracy: 0.1241 - val_loss: 3.0187 - val_top3_acc: 0.3138 - learning_rate: 0.0010 - lr: 0.0010
Epoch 3/200
150/150 ━━━━━━━━━━━━━━━━━━━━ 32s 215ms/step - accuracy: 0.3283 - loss: 2.6345 - top3_acc: 0.6050 - val_accuracy: 0.0931 - val_loss: 3.0950 - val_top3_acc: 0.2966 - learning_rate: 0.0010 - lr: 0.0010
Epoch 4/200
150/150 ━━━━━━━━━━━━━━━━━━━━ 34s 228ms/step - accuracy: 0.4142 - loss: 2.4158 - top3_acc: 0.7192 - val_accuracy: 0.1103 - val_loss: 2.9713 - val_top3_acc: 0.2793 - learning_rate: 0.0010 - lr: 0.0010
Epoch 5/200
150/150 ━━━━━━━━━━━━━━━━━━━━ 33s 223ms/step - accuracy: 0.5225 - loss: 2.1972 - top3_acc: 0.7983 - val_accuracy: 0.1241 - val_loss: 2.9611 - val

## 6 · Training curves

In [9]:
hist = history.history
epochs_ran = range(1, len(hist['loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'Training curves — {EXP_NAME}', fontsize=14)

axes[0].plot(epochs_ran, hist['loss'],     label='Train loss')
axes[0].plot(epochs_ran, hist['val_loss'], label='Val loss')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(epochs_ran, hist['accuracy'],     label='Train acc')
axes[1].plot(epochs_ran, hist['val_accuracy'], label='Val acc')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True)

if 'lr' in hist:
    axes[2].semilogy(epochs_ran, hist['lr'], color='green')
    axes[2].set_title('Learning Rate')
    axes[2].set_xlabel('Epoch')
    axes[2].grid(True)

plt.tight_layout()
curves_path = EXP_DIR / 'training_curves.png'
plt.savefig(curves_path, dpi=150, bbox_inches='tight')
plt.show()
print('Saved:', curves_path)

Saved: C:\Users\Napat\termtemsl\experiment_model\models_02\-3_8_oldprocessed_lstm_0.2_0.001_0.05_128-64-32\training_curves.png


## 7 · Evaluate on validation set

In [10]:
y_val_pred_prob = model.predict(X_val, batch_size=BATCH_SIZE, verbose=0)
y_val_pred      = np.argmax(y_val_pred_prob, axis=1)
y_val_true      = y_val.astype(int)

prec, rec, f1, sup = precision_recall_fscore_support(
    y_val_true, y_val_pred, average=None, labels=list(range(NUM_CLASSES))
)
prec_m, rec_m, f1_m, _ = precision_recall_fscore_support(
    y_val_true, y_val_pred, average='macro'
)
acc_val = float(np.mean(y_val_pred == y_val_true))

top3_val = top_k_accuracy_score(y_val_true, y_val_pred_prob, k=3)

METRICS = {
    'accuracy':       round(acc_val, 4),
    'top3_accuracy':  round(top3_val, 4),
    'macro_precision': round(prec_m, 4),
    'macro_recall':    round(rec_m,  4),
    'macro_f1':        round(f1_m,   4),
    'per_class': {
        CLASSES[i]: {
            'precision': round(float(prec[i]), 4),
            'recall':    round(float(rec[i]),  4),
            'f1':        round(float(f1[i]),   4),
            'support':   int(sup[i]),
        }
        for i in range(NUM_CLASSES)
    },
}

print(f'Val accuracy : {acc_val:.4f}')
print(f'Top-3 acc    : {top3_val:.4f}')
print(f'Macro F1     : {f1_m:.4f}')
print()
print(classification_report(y_val_true, y_val_pred, target_names=CLASSES))

Val accuracy : 0.2931
Top-3 acc    : 0.5552
Macro F1     : 0.2279

              precision    recall  f1-score   support

       ขอโทษ       0.57      0.28      0.37        29
          ดี       0.30      0.59      0.40        29
      สวัสดี       0.00      0.00      0.00        29
        อะไร       0.60      0.10      0.18        29
        เงิน       0.55      1.00      0.71        29
    โทรศัพท์       0.14      0.10      0.12        29
         ใคร       0.17      0.69      0.27        29
         ใช่       0.24      0.14      0.17        29
         ไม่       0.00      0.00      0.00        29
       ไม่ดี       0.33      0.03      0.06        29

    accuracy                           0.29       290
   macro avg       0.29      0.29      0.23       290
weighted avg       0.29      0.29      0.23       290



In [11]:
cm = confusion_matrix(y_val_true, y_val_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, data, title, fmt in zip(
    axes,
    [cm, cm_norm],
    ['Confusion Matrix (counts)', 'Confusion Matrix (normalised)'],
    ['d', '.2f'],
):
    sns.heatmap(
        data, ax=ax,
        annot=True, fmt=fmt, cmap='Blues',
        xticklabels=CLASSES, yticklabels=CLASSES,
        linewidths=0.5,
    )
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=45)

plt.suptitle(f'Confusion Matrix — {EXP_NAME}', fontsize=14)
plt.tight_layout()
cm_path = EXP_DIR / 'confusion_matrix.png'
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.show()
print('Saved:', cm_path)

Saved: C:\Users\Napat\termtemsl\experiment_model\models_02\-3_8_oldprocessed_lstm_0.2_0.001_0.05_128-64-32\confusion_matrix.png


In [12]:
df_pc = pd.DataFrame({
    'Class':     CLASSES,
    'Precision': prec,
    'Recall':    rec,
    'F1':        f1,
})

x = np.arange(NUM_CLASSES)
w = 0.25

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - w, df_pc['Precision'], w, label='Precision')
ax.bar(x,     df_pc['Recall'],    w, label='Recall')
ax.bar(x + w, df_pc['F1'],        w, label='F1')
ax.set_xticks(x)
ax.set_xticklabels(CLASSES, rotation=45, ha='right')
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title(f'Per-class Metrics — {EXP_NAME}')
ax.legend()
ax.grid(axis='y', alpha=0.4)

plt.tight_layout()
pc_path = EXP_DIR / 'per_class_metrics.png'
plt.savefig(pc_path, dpi=150, bbox_inches='tight')
plt.show()
print('Saved:', pc_path)

Saved: C:\Users\Napat\termtemsl\experiment_model\models_02\-3_8_oldprocessed_lstm_0.2_0.001_0.05_128-64-32\per_class_metrics.png


## 8 · Analysis

In [13]:
def analyze_results(cm, prec, rec, f1, hist):
    """Return structured analysis dict."""
    cm_offdiag = cm.copy()
    np.fill_diagonal(cm_offdiag, 0)
    confused_pairs = []
    flat_idx = np.argsort(cm_offdiag.ravel())[::-1][:10]
    for idx in flat_idx:
        r, c = divmod(int(idx), NUM_CLASSES)
        if cm_offdiag[r, c] > 0:
            confused_pairs.append({'true': CLASSES[r], 'pred': CLASSES[c], 'count': int(cm_offdiag[r, c])})

    sorted_f1   = sorted(enumerate(f1),  key=lambda x: x[1])
    weakest     = [CLASSES[i] for i, _ in sorted_f1[:3]]
    strongest   = [CLASSES[i] for i, _ in sorted_f1[-3:][::-1]]

    pr_imbalance = [
        {'class': CLASSES[i], 'precision': round(float(prec[i]), 3), 'recall': round(float(rec[i]), 3)}
        for i in range(NUM_CLASSES) if abs(prec[i] - rec[i]) > 0.15
    ]

    best_val  = max(hist.get('val_accuracy', [0]))
    final_train = hist.get('accuracy', [0])[-1]
    gap = final_train - best_val
    overfit   = gap > 0.15
    underfit  = best_val < 0.60

    return {
        'most_confused_pairs': confused_pairs,
        'weakest_classes':     weakest,
        'strongest_classes':   strongest,
        'pr_imbalance':        pr_imbalance,
        'overfitting':         {'detected': overfit,  'train_val_gap': round(float(gap), 4)},
        'underfitting':        {'detected': underfit, 'best_val_acc':  round(float(best_val), 4)},
    }


ANALYSIS = analyze_results(cm, prec, rec, f1, hist)

print('Most confused pairs:')
for p in ANALYSIS['most_confused_pairs'][:5]:
    print(f"  {p['true']} → {p['pred']}  ({p['count']} times)")
print('Weakest classes  :', ANALYSIS['weakest_classes'])
print('Strongest classes:', ANALYSIS['strongest_classes'])
if ANALYSIS['pr_imbalance']:
    print('P/R imbalance    :', [x['class'] for x in ANALYSIS['pr_imbalance']])
print('Overfitting  :', ANALYSIS['overfitting'])
print('Underfitting :', ANALYSIS['underfitting'])

Most confused pairs:
  ไม่ → ใคร  (26 times)
  สวัสดี → ใคร  (22 times)
  ใช่ → ดี  (21 times)
  ขอโทษ → ใคร  (21 times)
  โทรศัพท์ → ใคร  (20 times)
Weakest classes  : ['สวัสดี', 'ไม่', 'ไม่ดี']
Strongest classes: ['เงิน', 'ดี', 'ขอโทษ']
P/R imbalance    : ['ขอโทษ', 'ดี', 'อะไร', 'เงิน', 'ใคร', 'ไม่ดี']
Overfitting  : {'detected': True, 'train_val_gap': 0.6761}
Underfitting : {'detected': True, 'best_val_acc': 0.2931}


## 9 · Quick inference test

In [14]:
N_SAMPLES = 8
rng_idx   = np.random.choice(len(X_val), N_SAMPLES, replace=False)

X_sample  = X_val[rng_idx]
y_sample  = y_val_true[rng_idx]
probs     = model.predict(X_sample, verbose=0)
preds     = np.argmax(probs, axis=1)
confs     = probs[np.arange(N_SAMPLES), preds]

INFERENCE_RESULTS = []
for i in range(N_SAMPLES):
    entry = {
        'index':      int(rng_idx[i]),
        'ground_truth': idx_to_label[y_sample[i]],
        'predicted':    idx_to_label[preds[i]],
        'confidence':   round(float(confs[i]), 4),
        'correct':      bool(y_sample[i] == preds[i]),
    }
    INFERENCE_RESULTS.append(entry)
    status = '✓' if entry['correct'] else '✗'
    print(f"{status} [{i}] GT={entry['ground_truth']:12s}  Pred={entry['predicted']:12s}  Conf={entry['confidence']:.3f}")

if INPUT_TYPE == 'cutting':
    fig, axes = plt.subplots(N_SAMPLES, 5, figsize=(15, 3 * N_SAMPLES))
    for i in range(N_SAMPLES):
        frame_idx = np.linspace(0, X_sample.shape[1] - 1, 5).astype(int)
        for j, fi in enumerate(frame_idx):
            axes[i, j].imshow(X_sample[i, fi])
            axes[i, j].axis('off')
            if j == 0:
                c = 'green' if INFERENCE_RESULTS[i]['correct'] else 'red'
                axes[i, j].set_ylabel(
                    f"GT:{INFERENCE_RESULTS[i]['ground_truth']}\nPred:{INFERENCE_RESULTS[i]['predicted']}",
                    color=c, fontsize=7, rotation=0, labelpad=60
                )
    plt.suptitle('Inference — video frames', fontsize=12)
else:
    fig, axes = plt.subplots(N_SAMPLES, 1, figsize=(16, 2 * N_SAMPLES), sharex=True)
    for i, ax in enumerate(axes):
        seq = X_sample[i]
        motion = np.linalg.norm(seq, axis=1)
        ax.plot(motion, linewidth=0.8)
        c = 'green' if INFERENCE_RESULTS[i]['correct'] else 'red'
        ax.set_ylabel(INFERENCE_RESULTS[i]['ground_truth'], fontsize=7, color=c)
        ax.set_title(
            f"Pred: {INFERENCE_RESULTS[i]['predicted']}  conf={INFERENCE_RESULTS[i]['confidence']:.2f}",
            fontsize=7, color=c
        )
        ax.tick_params(labelsize=6)
    plt.suptitle('Inference — temporal keypoint norm', fontsize=12)

plt.tight_layout()
inf_vis_path = EXP_DIR / 'inference_validation.png'
plt.savefig(inf_vis_path, dpi=120, bbox_inches='tight')
plt.show()

✗ [0] GT=ขอโทษ         Pred=ใคร           Conf=0.960
✓ [1] GT=โทรศัพท์      Pred=โทรศัพท์      Conf=0.643
✗ [2] GT=ใช่           Pred=ดี            Conf=0.250
✗ [3] GT=สวัสดี        Pred=ใคร           Conf=0.865
✗ [4] GT=ไม่           Pred=ใคร           Conf=0.911
✗ [5] GT=ใคร           Pred=โทรศัพท์      Conf=0.435
✗ [6] GT=โทรศัพท์      Pred=ใคร           Conf=0.931
✗ [7] GT=ใช่           Pred=ดี            Conf=0.390


## 10 · Save model, artefacts & logs

In [15]:
saved_model_path = str(EXP_DIR / 'saved_model.keras')
model.save(saved_model_path)
print('SavedModel →', saved_model_path)

arch_path = EXP_DIR / 'architecture.json'
arch_path.write_text(model.to_json(indent=2))
print('Architecture →', arch_path)

le_path = EXP_DIR / 'label_encoder.json'
le_path.write_text(json.dumps(label_info, ensure_ascii=False, indent=2))
print('Label encoder →', le_path)

SavedModel → C:\Users\Napat\termtemsl\experiment_model\models_02\-3_8_oldprocessed_lstm_0.2_0.001_0.05_128-64-32\saved_model.keras
Architecture → C:\Users\Napat\termtemsl\experiment_model\models_02\-3_8_oldprocessed_lstm_0.2_0.001_0.05_128-64-32\architecture.json
Label encoder → C:\Users\Napat\termtemsl\experiment_model\models_02\-3_8_oldprocessed_lstm_0.2_0.001_0.05_128-64-32\label_encoder.json


In [16]:
CONFIG = {
    'exp_name':     EXP_NAME,
    'input_type':   INPUT_TYPE,
    'model_type':   MODEL_TYPE,
    'batch_size':   BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'num_classes':  NUM_CLASSES,
    'input_shape':  list(INPUT_SHAPE),
    'seed':         SEED,
    'timestamp':    datetime.datetime.now().isoformat(),
}
HYPERPARAMS = {
    'lstm_units':      LSTM_UNITS,
    'dropout_rate':    DROPOUT_RATE,
    'l2_reg':          L2_REG,
    'label_smoothing': LABEL_SMOOTHING,
    'mixup_alpha':     MIXUP_ALPHA,
    'epochs_total':    EPOCHS,
    'epochs_ran':      len(hist['loss']),
}

(EXP_DIR / 'config.json').write_text(json.dumps(CONFIG,      ensure_ascii=False, indent=2))
(EXP_DIR / 'hyperparams.json').write_text(json.dumps(HYPERPARAMS, ensure_ascii=False, indent=2))
(EXP_DIR / 'metrics.json').write_text(json.dumps(METRICS,    ensure_ascii=False, indent=2))
(EXP_DIR / 'inference_results.json').write_text(json.dumps(INFERENCE_RESULTS, ensure_ascii=False, indent=2))
(EXP_DIR / 'analysis_summary.json').write_text(json.dumps(ANALYSIS, ensure_ascii=False, indent=2))

print('JSON artefacts saved to', EXP_DIR)

JSON artefacts saved to C:\Users\Napat\termtemsl\experiment_model\models_02\-3_8_oldprocessed_lstm_0.2_0.001_0.05_128-64-32


In [17]:
import platform

gpu_info = []
for g in tf.config.list_physical_devices('GPU'):
    try:
        details = tf.config.experimental.get_device_details(g)
        gpu_info.append(details.get('device_name', g.name))
    except Exception:
        gpu_info.append(g.name)

LOG_ENTRY = {
    'exp_name':         EXP_NAME,
    'timestamp':        CONFIG['timestamp'],
    'tensorflow_version': tf.__version__,
    'platform':         platform.platform(),
    'gpu':              gpu_info or ['CPU only'],
    'training_time_min': round(TRAIN_TIME / 60, 2),
    'epochs_ran':       HYPERPARAMS['epochs_ran'],
    'val_accuracy':     METRICS['accuracy'],
    'top3_accuracy':    METRICS['top3_accuracy'],
    'macro_precision':  METRICS['macro_precision'],
    'macro_recall':     METRICS['macro_recall'],
    'macro_f1':         METRICS['macro_f1'],
    'lr_history':       [round(v, 8) for v in hist.get('lr', [])],
    'train_acc_history': [round(v, 4) for v in hist.get('accuracy', [])],
    'val_acc_history':   [round(v, 4) for v in hist.get('val_accuracy', [])],
}

LOG_FILE.write_text(json.dumps(LOG_ENTRY, ensure_ascii=False, indent=2))
print('Log →', LOG_FILE)

print('\n=== Experiment complete ===')
print(f'  Experiment : {EXP_NAME}')
print(f'  Val acc    : {METRICS["accuracy"]:.4f}')
print(f'  Macro F1   : {METRICS["macro_f1"]:.4f}')
print(f'  Saved to   : {EXP_DIR}')

Log → C:\Users\Napat\termtemsl\experiment_model\logs\training\-3_8_oldprocessed_lstm_0.2_0.001_0.05_128-64-32.json

=== Experiment complete ===
  Experiment : -3_8_oldprocessed_lstm_0.2_0.001_0.05_128-64-32
  Val acc    : 0.2931
  Macro F1   : 0.2279
  Saved to   : C:\Users\Napat\termtemsl\experiment_model\models_02\-3_8_oldprocessed_lstm_0.2_0.001_0.05_128-64-32
